# STAT 764 · Meeting 5 — Regression as prediction

**Tuesday, September 22**

You have fitted regressions before. What changes when the goal is *predicting new
cases* rather than describing the ones you have?

Three things, and none of them are about the mathematics of least squares:

1. **The baseline moves first.** Before any model, what does doing nothing get you?
2. **The metric is a decision.** R-squared is not the only answer, and often not the
   useful one.
3. **The scale you work on is also a decision** — and it can quietly change which
   model looks best.

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

import numpy as np
import pandas as pd
from stat764 import load

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ames = load("ames.csv")

NUMERIC = ["Gr_Liv_Area", "Year_Built", "Overall_Qual", "Total_Bsmt_SF", "Lot_Area"]
CATEGORICAL = ["Neighborhood", "Central_Air"]

X = ames[NUMERIC + CATEGORICAL]
y = ames["SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=764)


def pipeline(model=None):
    """The Meeting 2 skeleton, unchanged. It does not change all semester."""
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                              ("scale", StandardScaler())]), NUMERIC),
            ("cat", Pipeline([("fill", SimpleImputer(strategy="constant",
                                                     fill_value="Missing")),
                              ("encode", OneHotEncoder(handle_unknown="ignore"))]),
             CATEGORICAL),
        ])),
        ("model", model or LinearRegression()),
    ])

## 1. Start from doing nothing

Before any model: predict the training mean for every house. This is not a straw man —
it is the number your model has to beat to have earned its existence.

In [ ]:
baseline = np.full(len(y_test), y_train.mean())

print(f"  predict ${y_train.mean():,.0f} for every house")
print(f"    R-squared {r2_score(y_test, baseline):>8.3f}")
print(f"    RMSE      ${root_mean_squared_error(y_test, baseline):>10,.0f}")
print(f"    MAE       ${mean_absolute_error(y_test, baseline):>10,.0f}")

R-squared of about zero, by construction — that is what R-squared *means*. But look at
the other two numbers: the average house is off by roughly **$62,500**.

That is the number a homeowner would care about, and the R-squared does not tell you it.

## 2. The metric is a decision

Now fit the model, and report all three.

In [ ]:
fit = pipeline().fit(X_train, y_train)
pred = fit.predict(X_test)

rows = []
for name, p in [("mean baseline", baseline), ("OLS pipeline", pred)]:
    rows.append({"model": name,
                 "R2": r2_score(y_test, p),
                 "RMSE $": root_mean_squared_error(y_test, p),
                 "MAE $": mean_absolute_error(y_test, p)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

**RMSE and MAE disagree about how good this is**, and the difference is informative.

RMSE squares the errors before averaging, so it is dominated by the houses you got
badly wrong. MAE treats a $10,000 miss as exactly ten times a $1,000 miss. If you are
advising a homeowner, MAE is closer to what they mean by "how far off are you". If you
are worried about catastrophic misvaluations, RMSE is.

**Neither is correct. The question is what decision the number is for** — which is a
theme for the rest of the course, and returns properly in Week 6 with classification.

## 3. The scale is a decision too

House prices are right-skewed: a few very expensive houses stretch the tail. The
standard advice is to model `log(price)`.

In [ ]:
print(f"  skew of SalePrice       {ames['SalePrice'].skew():>6.2f}")
print(f"  skew of log(SalePrice)  {np.log(ames['SalePrice']).skew():>6.2f}")

On that evidence the transformation looks obviously right — it turns a badly skewed
outcome into an almost perfectly symmetric one.

So fit on the log scale, then convert the predictions back to dollars and score them.

In [ ]:
log_fit = pipeline().fit(X_train, np.log(y_train))
log_pred = np.exp(log_fit.predict(X_test))

rows.append({"model": "OLS on log(price), back to $",
             "R2": r2_score(y_test, log_pred),
             "RMSE $": root_mean_squared_error(y_test, log_pred),
             "MAE $": mean_absolute_error(y_test, log_pred)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

**It got worse.** Substantially worse on R-squared and RMSE — and yet slightly *better*
on MAE.

That is not a bug, and it is worth sitting with. Two things are happening:

- Fitting on the log scale minimises error *in log space*, which means it cares about
  **proportional** error. A 10% miss on a $400,000 house counts the same as a 10% miss
  on a $100,000 house. On the dollar scale, those are $40,000 and $10,000.
- `exp()` of a prediction of the mean log is not the mean price. It is closer to the
  median, so the predictions are systematically low.

MAE improves — $20,514 against $22,718 — because typical houses get predicted better.
RMSE collapses, $56,816 against $38,987, because the expensive ones get much worse and
RMSE cares about exactly those.

⚠ **The transformation was not wrong. The evaluation changed what "better" means.** If
you report percentage error, the log model wins. If you report dollars, it loses. You
have to choose, and say which you chose.

## 4. Imputation belongs inside — and now it matters

In Meeting 2 the leak from preprocessing outside the pipeline cost exactly nothing,
because scaling cannot change an OLS fit. Imputation is different: it invents values,
and the value it invents depends on which rows it saw.

In [ ]:
LOTS_MISSING = ["Lot_Frontage", "Mas_Vnr_Area", "Garage_Yr_Blt"]
print("  missing rates:")
for c in LOTS_MISSING:
    print(f"    {c:<18}{ames[c].isna().mean():>7.1%}")

wide_numeric = NUMERIC + LOTS_MISSING
Xw = ames[wide_numeric + CATEGORICAL]
Xw_train, Xw_test, yw_train, yw_test = train_test_split(
    Xw, y, test_size=0.25, random_state=764)

honest = Pipeline([
    ("prep", ColumnTransformer([
        ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), wide_numeric),
        ("cat", Pipeline([("fill", SimpleImputer(strategy="constant",
                                                 fill_value="Missing")),
                          ("encode", OneHotEncoder(handle_unknown="ignore"))]),
         CATEGORICAL),
    ])),
    ("model", LinearRegression()),
]).fit(Xw_train, yw_train)

print(f"\n  medians the pipeline learned, from the TRAINING rows only:")
med = honest.named_steps["prep"].named_transformers_["num"].named_steps["fill"].statistics_
for c, m in zip(wide_numeric, med):
    print(f"    {c:<18}{m:>10,.1f}")
print(f"\n  R-squared on unseen data: {r2_score(yw_test, honest.predict(Xw_test)):.3f}")

Those medians are parameters. They were estimated, from data, and the pipeline
estimated them from the training rows only — which is the entire reason it exists.

Meeting 22 is a whole session on what else you could have put there and what each
choice costs you.

## Studio

Beat the baseline honestly, and report a number you would defend to a homeowner.

1. Build a pipeline that beats the mean baseline on **MAE in dollars**, not R-squared.
2. Try at least one transformation — of the outcome, or of a skewed predictor — and
   measure whether it helps **on the metric you chose**.
3. Report three numbers: R-squared, RMSE, MAE. Be ready to say which one you would put
   in front of a client, and why.

In [ ]:
# YOUR CODE HERE
#
# my_pipe = pipeline(...)
# ...

## Compare

1. Whose model has the lowest MAE? Is it the same model as the lowest RMSE?
2. Did a transformation help you? On which metric?
3. If a homeowner asked "how far off will you be", what number would you say?

## Exit ticket

> You reported three numbers for the same model. Which one would you put in a report,
> and what does choosing it hide?

## Also in this neighborhood

📗 **Quantile regression.** If you care about "how far off am I, typically", modelling
the median directly is more honest than modelling the mean and reporting MAE.

📗 **Duan's smearing estimator.** The standard correction for the back-transform bias
in §3. Worth knowing that the problem has a name and a fix.

🚫 **Reporting R-squared alone.** It is scale-free, which is convenient, and it is
scale-free, which means it cannot tell you whether being wrong costs $5,000 or $50,000.